# Notebook 02: Sequential Tokenization via Fixed-Width N-Gram Slicing

## 1.1 The Tokenization Challenge in Binary Vectors
Unlike standard Natural Language Processing (NLP) domains where text streams are explicitly divided by spaces, binary Industrial Control Protocols (ICPs) present continuous, spaceless byte streams. Tokenization acts as the mathematical bridge that groups contiguous bytes into multi-byte candidate fields based on spatial co-occurrence.

## 1.2 Mathematical Model: Fixed-Width N-Grams
We define an $n$-gram token $T$ extracted from a packet vector $\mathbf{x}_i$ at continuous column offset $j$ as an $n$-length sequence tuple:

$$T_{j}^{(n)} = (b_{i,j}, b_{i,j+1}, \dots, b_{i,j+n-1})$$

### Architectural Sizing Choices ($n=2, n=4$)
This engine implements a uniform sliding window where the parameter $n$ remains constant across the loop. In industrial networking architectures, $n=2$ (16-bit) and $n=4$ (32-bit) configurations are highly representative because they map directly to hardware memory alignments (e.g., Modbus register banks, DNP3 block sizes, and 16-bit command opcodes).

In [8]:
import numpy as np
import os

# Define the path to the cached intermediate data artifact
PROCESSED_MATRIX_PATH = os.path.join("..", "data", "processed", "vectorized_matrix.npy")

if not os.path.exists(PROCESSED_MATRIX_PATH):
    raise FileNotFoundError(
        f"Could not find cached matrix at {PROCESSED_MATRIX_PATH}. "
        "Please run Notebook 01 to completion to generate and save this file first!"
    )

# Load the array instantly from disk memory
X = np.load(PROCESSED_MATRIX_PATH)
print(f"Real data tensor matrix successfully loaded from cache.")
print(f"Dimensions available for Tokenization: {X.shape[0]} Packets (N) x {X.shape[1]} Columns (M)")

Real data tensor matrix successfully loaded from cache.
Dimensions available for Tokenization: 1275 Packets (N) x 42 Columns (M)


## 2.1 Implementing the Structural Tokenization Engine
The following class handles sliding window extraction across our 2D tensor matrix. Converting the slices into Python `tuples` ensures they are immutable and hashable, which allows us to perform fast global frequency counting in downstream modules.

In [11]:
class StructuralTokenizationEngine:
    """
    Extracts structured sequential token mappings from an optimized 2D tensor matrix
    using an overlapping sliding window approach.
    """
    def __init__(self, window_size: int = 2):
        self.n = window_size
        
    def extract_corpus_tokens(self, matrix: np.ndarray) -> list:
        """
        Slides an n-byte window horizontally across rows within a payload matrix.
        
        Parameters:
        -----------
        matrix : np.ndarray
            The pre-allocated target numerical tensor of shape (N packets x M bytes).
            
        Returns:
        --------
        list of lists
            A sequential structure where each nested list preserves the 
            ordered token stream corresponding to an individual packet.
        """
        N, M = matrix.shape
        if M < self.n:
            raise ValueError(f"Window sizing (n={self.n}) exceeds matrix column bounds (M={M}).")
            
        global_token_stream = []
        
        for i in range(N):
            packet_tokens = []
            # Slide window across the individual sequence columns
            for j in range(M - self.n + 1):
                # Isolate sub-vector slicing segment
                token_slice = matrix[i, j:j+self.n]
                # Convert to immutable tuple to allow downstream hashing and counting
                packet_tokens.append(tuple(token_slice))
                
            global_token_stream.append(packet_tokens)
            
        return global_token_stream

## 3.1 Slicing Engine Execution (Bigram Configuration)
We initialize our slicing engine with $n=2$ (Bigram mode) to match standard 16-bit industrial register boundaries.

In [12]:
# Initialize tokenizer for 16-bit / 2-byte field analysis
tokenizer = StructuralTokenizationEngine(window_size=2)
tokenized_corpus = tokenizer.extract_corpus_tokens(X)

print("--- EXTRACED TOKEN GEOMETRY ---")
print(f"Total processed packets: {len(tokenized_corpus)}")
print(f"Tokens extracted per packet: {len(tokenized_corpus[0])} (Calculated as M - n + 1)")

# Convert the first 5 tokens of packet 0 into a readable hexadecimal display format
formatted_sample = [[f"{byte:02x}" for byte in token] for token in tokenized_corpus[0][:5]]
print(f"\nFirst 5 extracted Bigram tokens (Packet 0, Hex format):\n{formatted_sample}")

--- EXTRACED TOKEN GEOMETRY ---
Total processed packets: 1275
Tokens extracted per packet: 41 (Calculated as M - n + 1)

First 5 extracted Bigram tokens (Packet 0, Hex format):
[['00', '01'], ['01', '08'], ['08', '00'], ['00', '06'], ['06', '04']]


## 4.1 Structural Inspection Matrix
To verify that our overlapping sliding windows are aligned properly, we load the extracted tokens into a structural inspection DataFrame. This helps us ensure that no data fields were skipped during the slicing loops.

In [13]:
# Map packet tokens over their actual structural offsets
offset_labels = [f"Offset {i}-{i+1}" for i in range(len(tokenized_corpus[0]))]
hex_tokens_p0 = [" ".join([f"{b:02x}" for b in tok]) for tok in tokenized_corpus[0]]
hex_tokens_p1 = [" ".join([f"{b:02x}" for b in tok]) for tok in tokenized_corpus[1]]

inspection_df = pd.DataFrame({
    "Packet 0 Real Tokens": hex_tokens_p0,
    "Packet 1 Real Tokens": hex_tokens_p1
}, index=offset_labels)

# Display a clean subset slice to verify continuous step processing
inspection_df.head(14)

,Packet 0 Real Tokens,Packet 1 Real Tokens
Offset 0-1,00 01,00 01
Offset 1-2,01 08,01 08
Offset 2-3,08 00,08 00
Offset 3-4,00 06,00 06
Offset 4-5,06 04,06 04
Offset 5-6,04 00,04 00
Offset 6-7,00 01,00 02
Offset 7-8,01 00,02 00
Offset 8-9,00 50,00 50
Offset 9-10,50 56,50 56


In [14]:
import pickle
import os

processed_dir = os.path.join("..", "data", "processed")
tokens_cache_path = os.path.join(processed_dir, "tokenized_corpus.pkl")

# Serialize the nested token sequences to a binary file
with open(tokens_cache_path, "wb") as f:
    pickle.dump(tokenized_corpus, f)

print(f"Success! Tokenized corpus cached for Notebook 03 at:\n{tokens_cache_path}")

Success! Tokenized corpus cached for Notebook 03 at:
..\data\processed\tokenized_corpus.pkl


## 5.1 Supervisor Demonstration Guide

### Key Takeaways to Highlight during the Demo:
* **Overlapping Structural Capture:** Explain that overlapping windows guarantee that every potential boundary alignment is evaluated. For instance, notice how `Offset 0-1` captures `00 01` (Hardware Type) and `Offset 1-2` immediately captures `01 08`. This step captures the transition boundary between fields without requiring any pre-labeled protocol schemas.
* **Hashability for Frequency Counts:** Point out that transforming slices into immutable Python tuples prepares the dataset for fast global frequency profiling in the next notebook.

### Anticipated Supervisor Defenses:
* **Q:** *Why rely on overlapping sliding windows instead of non-overlapping chunks, which would use less processing time and memory?*
* **A:** Non-overlapping chunks require you to know the exact field boundaries beforehand. If your starting alignment is off by even a single byte, a non-overlapping slice will split a single logical field (such as a 2-byte Opcode) across two separate blocks. Overlapping windows ensure that every potential field alignment is captured, allowing our downstream statistical filters to isolate the true boundaries.